# A2 corregido: Random Forest (correcciones i, ii, iii y v)

Correr con este notebook en la misma carpeta que `metadata.csv`, `taxonomy_relabd.species.csv`,
`comun.py` y `pliegues_por_paciente.csv`.

- **(i, ii)** En lugar de `train_test_split` por muestra se usan los pliegues comunes por paciente
  (5 pliegues x 5 repeticiones). Ninguna paciente tiene muestras en entrenamiento y prueba a la vez.
  La probabilidad de cada paciente es el promedio de sus muestras.
- **(iii)** El umbral (Youden) se calcula solo con las pacientes de entrenamiento (`umbral_interno`):
  cada pliegue entrena 4 bosques (3 internos + el final).
- **(v)** Al final, el mismo Random Forest con la abundancia media de cada paciente (sin orden temporal).

Hiperparámetros iguales al original (100 árboles, sin límite de profundidad, `class_weight='balanced'`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

from comun import (RESULTADOS, cargar_pliegues, iterar_pliegues, promedio_por_paciente,
                   umbral_interno, metricas)
RESULTADOS.mkdir(exist_ok=True)

metadata = pd.read_csv("metadata.csv")
taxonomy = pd.read_csv("taxonomy_relabd.species.csv")


meta_subset = metadata[['specimen', 'participant_id', 'was_preterm']].copy()


full_data = pd.merge(meta_subset, taxonomy, on='specimen')


X = full_data.drop(columns=['specimen', 'participant_id', 'was_preterm'])
y = full_data['was_preterm'].astype(int).values
pacientes = full_data['participant_id'].values

print(X.shape, '| pacientes:', len(set(pacientes)))

In [ ]:
def entrenar_rf(X_train, y_train, X_test):
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    return rf_model.predict_proba(X_test)[:, 1]


pliegues = cargar_pliegues()
filas, predicciones, importancias = [], [], []
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes, list(ids_tr)), np.isin(pacientes, list(ids_te))

    # Umbral con datos de entrenamiento solamente (punto iii)
    umbral = umbral_interno(entrenar_rf, X.values[tr], y[tr], pacientes[tr], semilla=100 + 10 * rep + fold)

    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X[tr], y[tr])
    y_prob = rf_model.predict_proba(X[te])[:, 1]

    pp = promedio_por_paciente(pacientes[te], y[te], y_prob)
    filas.append(dict(modelo='Random Forest', rep=rep, fold=fold, **metricas(pp.y, pp.prob, umbral)))
    predicciones.append(pp.assign(rep=rep, fold=fold))
    importancias.append(rf_model.feature_importances_)
    print(f'rep {rep} fold {fold} listo')

pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A2.csv', index=False)
res = pd.DataFrame(filas)
res.to_csv(RESULTADOS / 'metricas_A2.csv', index=False)

print("--- Random Forest Results (25 pliegues por paciente) ---")
print(res[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3))

In [ ]:
# Importancia media de cada especie en los 25 pliegues
feature_imp_df = pd.DataFrame({'Bacteria': X.columns, 'Importance': np.mean(importancias, axis=0),
                               'Std': np.std(importancias, axis=0)})
top_10_bacteria = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)
top_10_bacteria.to_csv(RESULTADOS / 'top10_importancia_A2.csv', index=False)

print("\n--- Top 10 Bacterias Predictoras ---")
print(top_10_bacteria)


plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Bacteria', hue='Bacteria', data=top_10_bacteria, palette='viridis', legend=False)
plt.title('Top 10 Biomarcadores Bacterianos (importancia media en 25 pliegues por paciente)')
plt.show()

## Punto v: modelo estático de la media por paciente

Control para la interpretación dinámica de la CNN-LSTM. Cada paciente se resume en **un solo renglón**:
la abundancia relativa media de cada especie sobre todas sus muestras, sin orden temporal. Mismo Random
Forest, mismos 25 pliegues y mismo umbral interno. Si la CNN-LSTM no supera a este modelo, la trayectoria
no aporta información más allá de la composición promedio. Guarda `metricas_A2_media.csv` y
`pred_A2_media.csv` (no modifica `metricas_A2.csv`, que usa la Tabla 4.1).


In [ ]:
# Un renglón por paciente con la abundancia media de cada especie
X_media = full_data.drop(columns=['specimen', 'was_preterm']).groupby('participant_id').mean()
y_media = full_data.groupby('participant_id')['was_preterm'].first().astype(int).loc[X_media.index].values
pacientes_media = X_media.index.values
print(X_media.shape, '| pretérmino:', y_media.sum())

filas_media, pred_media = [], []
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes_media, list(ids_tr)), np.isin(pacientes_media, list(ids_te))
    umbral = umbral_interno(entrenar_rf, X_media.values[tr], y_media[tr], pacientes_media[tr],
                            semilla=100 + 10 * rep + fold)
    y_prob = entrenar_rf(X_media.values[tr], y_media[tr], X_media.values[te])
    pp = promedio_por_paciente(pacientes_media[te], y_media[te], y_prob)   # ya es una fila por paciente
    filas_media.append(dict(modelo='Random Forest (media por paciente)', rep=rep, fold=fold,
                            **metricas(pp.y, pp.prob, umbral)))
    pred_media.append(pp.assign(rep=rep, fold=fold))

pd.concat(pred_media).to_csv(RESULTADOS / 'pred_A2_media.csv', index=False)
res_media = pd.DataFrame(filas_media)
res_media.to_csv(RESULTADOS / 'metricas_A2_media.csv', index=False)

print("--- Random Forest, media por paciente (25 pliegues) ---")
print(res_media[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3))